# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# TF-IDF (Term Frequency – Inverse Document Frequency)
#
# La idea central: una palabra es importante en un documento si aparece mucho
# en ESE documento (TF alto) pero NO en todos los demás documentos (IDF alto).
#
#   TF(t, d)  = frecuencia del término t en el documento d
#   IDF(t)    = log( N / df(t) )  donde N = total de docs, df(t) = docs que contienen t
#   TF-IDF(t, d) = TF(t, d) × IDF(t)
#
# Resultado: una MATRIZ donde cada FILA es un documento representado como
# un vector numérico en el espacio del vocabulario.
# ─────────────────────────────────────────────────────────────────────────────

import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Cargar corpus ---
# El corpus tiene 500 líneas; cada línea es un documento independiente.
ruta_turismo = os.path.join(os.getcwd(), '01_corpus_turismo_500.txt')

with open(ruta_turismo, 'r', encoding='utf-8') as f:
    corpus_turismo = f.read().splitlines()  # lista de 500 strings

print(f'Número de documentos: {len(corpus_turismo)}')
print('Ejemplo de documento:')
print(corpus_turismo[0])

Número de documentos: 500
Ejemplo de documento:
Otavalo es conocido por su mercado indígena y su artesanía Perfecto para rafting.


In [2]:
# --- Crear y ajustar el vectorizador TF-IDF ---
#
# TfidfVectorizer de sklearn hace todo el trabajo:
#   1. Tokeniza cada documento (separa en palabras)
#   2. Construye el vocabulario (columnas de la matriz)
#   3. Calcula TF-IDF para cada par (documento, término)
#
# Parámetros clave:
#   - max_df=0.95 : ignora términos que aparecen en más del 95% de los docs
#                   (son tan comunes que no distinguen nada, como stopwords)
#   - min_df=2    : ignora términos que aparecen en menos de 2 documentos
#                   (tan raros que tampoco ayudan)

vectorizador_turismo = TfidfVectorizer(max_df=0.95, min_df=2)

# fit_transform hace dos cosas a la vez:
#   fit()      -> aprende el vocabulario y los valores IDF del corpus
#   transform()-> convierte cada documento a su vector TF-IDF
matriz_turismo = vectorizador_turismo.fit_transform(corpus_turismo)

# La matriz es 'sparse' (dispersa): guarda solo los valores != 0
# porque la mayoría de palabras no aparecen en cada documento.
print(f'Forma de la matriz TF-IDF: {matriz_turismo.shape}')
print(f'  → {matriz_turismo.shape[0]} documentos × {matriz_turismo.shape[1]} términos del vocabulario')
print(f'Densidad (valores != 0): {matriz_turismo.nnz / (matriz_turismo.shape[0] * matriz_turismo.shape[1]):.2%}')

Forma de la matriz TF-IDF: (500, 116)
  → 500 documentos × 116 términos del vocabulario
Densidad (valores != 0): 9.90%


In [3]:
# --- Explorar la matriz como DataFrame para visualizarla mejor ---
#
# Convertimos a DataFrame solo para exploración; en producción se trabaja
# con la matriz sparse porque ocupa mucho menos memoria.

vocabulario_turismo = vectorizador_turismo.get_feature_names_out()

df_turismo = pd.DataFrame(
    matriz_turismo.toarray(),       # .toarray() convierte sparse → numpy (¡consume más RAM!)
    columns=vocabulario_turismo
)

print('Primeras 5 filas y 10 columnas de la matriz TF-IDF:')
df_turismo.iloc[:5, :10]

Primeras 5 filas y 10 columnas de la matriz TF-IDF:


,2000,agua,amazonía,arquitectura,artesanía,atrae,atraen,auténtico,aventura,aves
0,0.0,0.0,0.0,0.0,0.341664,0.000000,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.352958,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0


In [4]:
# --- ¿Qué palabras tienen mayor TF-IDF promedio en el corpus? ---
# Las palabras con mayor promedio son las más 'discriminativas' del corpus.

tfidf_promedio = df_turismo.mean(axis=0).sort_values(ascending=False)

print('Top 15 términos por TF-IDF promedio:')
print(tfidf_promedio.head(15))

Top 15 términos por TF-IDF promedio:
el             0.091438
para           0.090165
su             0.078960
de             0.071858
un             0.066400
ideal          0.064379
feriado        0.061232
la             0.056321
por            0.055446
es             0.054663
próximo        0.052930
experiencia    0.047647
inolvidable    0.047647
una            0.047647
perfecto       0.047535
dtype: float64


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Construir el corpus Gutenberg
#
# En recuperación de información, un 'corpus' es la colección de documentos
# sobre la que vamos a hacer búsquedas. Aquí cada LIBRO es un documento.
#
# Usamos los libros de la carpeta '1000libros/' descargados con
# descargar_1000.py. Son 898 libros del Proyecto Gutenberg en español
# (102 no tienen versión .txt disponible en Gutendex).
# ─────────────────────────────────────────────────────────────────────────────

ruta_libros = os.path.join(os.getcwd(), '1000libros')
archivos = sorted(os.listdir(ruta_libros))  # lista de nombres de archivo

print(f'Libros disponibles: {len(archivos)}')

# Cargamos cada libro como un elemento de la lista 'corpus_gutenberg'.
# También guardamos los nombres de archivo en 'nombres_gutenberg'
# para poder mostrar qué libro corresponde a cada resultado de búsqueda.
corpus_gutenberg = []
nombres_gutenberg = []

for archivo in archivos:
    ruta_archivo = os.path.join(ruta_libros, archivo)
    try:
        with open(ruta_archivo, 'r', encoding='utf-8') as f:
            texto = f.read()
        corpus_gutenberg.append(texto)
        nombres_gutenberg.append(archivo)
    except Exception as e:
        print(f'  Error al leer {archivo}: {e}')

print(f'Libros cargados correctamente: {len(corpus_gutenberg)}')
print(f'Ejemplo: "{nombres_gutenberg[0]}" – {len(corpus_gutenberg[0])} caracteres')

Libros disponibles: 898
Libros cargados correctamente: 898
Ejemplo: "20 poemas para ser leídos en el tranvía.txt" – 41295 caracteres


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# TF-IDF sobre el corpus de libros
#
# Diferencia clave respecto al corpus de turismo:
#   - Aquí cada documento es un libro entero (cientos de miles de palabras)
#   - El vocabulario será MUCHO mayor
#   - La matriz tendrá pocas filas (un libro = una fila) pero muchas columnas
#
# Parámetros ajustados para libros grandes:
#   - max_features=50000 : limitamos el vocabulario a las 50,000 palabras
#     más frecuentes para no agotar la memoria
#   - sublinear_tf=True  : aplica log(1 + TF) en lugar de TF puro;
#     evita que documentos muy largos dominen solo por tener más palabras
# ─────────────────────────────────────────────────────────────────────────────

vectorizador_gutenberg = TfidfVectorizer(
    max_features=50000,
    sublinear_tf=True,    # log(1+TF) — más justo para documentos de distinto tamaño
    max_df=0.95,          # descarta términos en más del 95% de los libros
    min_df=2              # descarta términos que aparecen en un solo libro
)

# fit_transform aprende el vocabulario de todos los libros y construye la matriz.
# Esto puede tardar unos segundos con 100 libros grandes.
import time
t0 = time.time()
matriz_gutenberg = vectorizador_gutenberg.fit_transform(corpus_gutenberg)
t1 = time.time()

print(f'Tiempo de vectorización: {t1 - t0:.2f} s')
print(f'Forma de la matriz: {matriz_gutenberg.shape}')
print(f'  → {matriz_gutenberg.shape[0]} libros × {matriz_gutenberg.shape[1]} términos')
print(f'Valores no cero: {matriz_gutenberg.nnz:,} ({matriz_gutenberg.nnz / (matriz_gutenberg.shape[0]*matriz_gutenberg.shape[1]):.2%} de densidad)')

Tiempo de vectorización: 30.91 s
Forma de la matriz: (898, 50000)
  → 898 libros × 50000 términos
Valores no cero: 6,606,560 (14.71% de densidad)


In [7]:
# --- Ver los 10 términos con mayor TF-IDF en un libro específico ---
#
# Para interpretar la matriz: el vector de un libro tiene un valor alto
# en los términos que son muy frecuentes en ese libro pero raros en los demás.
# Esos son los 'términos característicos' del libro.

import numpy as np

idx_libro = 0  # cambia este índice para explorar otros libros

vocabulario_gutenberg = vectorizador_gutenberg.get_feature_names_out()
vector_libro = matriz_gutenberg[idx_libro].toarray().flatten()

# Ordenamos los índices de mayor a menor TF-IDF
top_indices = np.argsort(vector_libro)[::-1][:10]

print(f'Términos más representativos de "{nombres_gutenberg[idx_libro]}":')
for i in top_indices:
    print(f'  {vocabulario_gutenberg[i]:<25} TF-IDF = {vector_libro[i]:.4f}')

Términos más representativos de "20 poemas para ser leídos en el tranvía.txt":
  1921                      TF-IDF = 0.1353
  1920                      TF-IDF = 0.1274
  vereda                    TF-IDF = 0.1006
  tranvía                   TF-IDF = 0.0988
  oliverio                  TF-IDF = 0.0952
  leídos                    TF-IDF = 0.0935
  pórfido                   TF-IDF = 0.0858
  nalgas                    TF-IDF = 0.0823
  1922                      TF-IDF = 0.0793
  croquis                   TF-IDF = 0.0788


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Búsqueda por similitud coseno con TF-IDF
#
# ¿Por qué similitud coseno?
#   Mide el ÁNGULO entre dos vectores, no su distancia euclidiana.
#   Valor entre 0 (sin relación) y 1 (idénticos en contenido).
#   Es robusta ante diferencias de longitud: un libro largo y uno corto
#   sobre el mismo tema tendrán coseno alto aunque sus vectores tengan
#   magnitudes muy distintas.
#
# Proceso de la búsqueda:
#   1. Transformar la consulta en un vector TF-IDF (usando el vectorizador
#      ya entrenado — importante: NO re-entrenar con la consulta)
#   2. Calcular la similitud coseno entre el vector-consulta y CADA fila
#      de la matriz (cada libro)
#   3. Ordenar los libros de mayor a menor similitud
#   4. Devolver los top-k más similares
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.metrics.pairwise import cosine_similarity

def buscar(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """
    Busca los libros más relevantes para una consulta usando TF-IDF + coseno.

    Args:
        query : texto de la consulta (ej. "amor y muerte")
        top_k : cuántos resultados devolver (por defecto 5)

    Returns:
        Lista de tuplas (nombre_libro, score) ordenadas de mayor a menor score.
    """
    # transform() (sin fit) convierte la consulta al mismo espacio vectorial
    # que ya aprendió el vectorizador. Las palabras desconocidas se ignoran.
    vector_query = vectorizador_gutenberg.transform([query])  # shape (1, vocabulario)

    # cosine_similarity devuelve una matriz (1 × N_libros)
    # con el score de similitud entre la consulta y cada libro
    scores = cosine_similarity(vector_query, matriz_gutenberg).flatten()

    # argsort da los índices de menor a mayor → invertimos con [::-1]
    indices_ordenados = np.argsort(scores)[::-1][:top_k]

    # Construimos la lista de resultados: (nombre_archivo, score)
    resultados = [(nombres_gutenberg[i], round(scores[i], 4)) for i in indices_ordenados]
    return resultados

In [9]:
# --- Probar la función buscar() ---

consultas = [
    "amor y muerte",
    "historia de América",
    "aventura y viaje",
    "poesía lírica",
]

for consulta in consultas:
    print(f'\nConsulta: "{consulta}"')
    print('-' * 50)
    for libro, score in buscar(consulta, top_k=3):
        print(f'  [{score:.4f}] {libro}')


Consulta: "amor y muerte"
--------------------------------------------------
  [0.0525] Místicas poesías.txt
  [0.0408] A Flor De Piel Frases.txt
  [0.0372] Lira Póstuma Obras Completas Vol. XXI.txt

Consulta: "historia de América"
--------------------------------------------------


  [0.0590] El derecho internacional americano estudio doctrinal y crítico.txt
  [0.0382] Canto a la Argentina, Oda a Mitre y otros poemas Obras Completas Vol. IX.txt
  [0.0372] España y los Estados Unidos de Norte América  $b a propósito de la guerra.txt

Consulta: "aventura y viaje"
--------------------------------------------------
  [0.0333] Doble error  $b Novela.txt
  [0.0297] Un año en quince minutos pieza en un acto.txt
  [0.0263] El libro de las mil noches y una noche t. 8.txt

Consulta: "poesía lírica"
--------------------------------------------------


  [0.0449] El corazón juglar.txt
  [0.0433] Canto a la Argentina, Oda a Mitre y otros poemas Obras Completas Vol. IX.txt
  [0.0407] Prosas Profanas Obras Completas Vol. II.txt


In [10]:
# --- Búsqueda interactiva ---
#
# Modifica la variable 'mi_consulta' y vuelve a ejecutar la celda
# para buscar en el corpus.

mi_consulta = "Don Quijote caballero"  # ← cambia aquí tu consulta

resultados = buscar(mi_consulta, top_k=5)

print(f'Resultados para: "{mi_consulta}"')
print('=' * 50)
for rank, (libro, score) in enumerate(resultados, start=1):
    print(f'{rank}. [{score:.4f}] {libro}')

Resultados para: "Don Quijote caballero"
1. [0.0536] El Consejo de los Dioses.txt
2. [0.0501] Vida de Don Quijote y Sancho.txt
3. [0.0500] O locura o santidad  $b Drama en tres actos y en prosa.txt
4. [0.0498] Tres novelas ejemplares y un prólogo.txt
5. [0.0493] Vida de Cervantes.txt
